# Predição da Resistência à Compressão do Concreto — MLP (PyTorch)

Etapa 1 do projeto: carregar o dataset, explorar rapidamente, separar treino/teste,
normalizar, e treinar uma MLP com validação cruzada (K-Fold) e early stopping.

Próximas etapas (ainda não incluídas aqui): baseline com XGBoost, SHAP, e otimização de formulação.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Carregar os dados

Baixe o `concrete.csv` do Kaggle (link do dataset) e coloque em `data/concrete.csv`.
Como não sabemos de antemão o nome exato das colunas nessa versão específica do dataset,
a primeira célula abaixo só lista as colunas — ajuste o dicionário de renomeação se precisar.

In [ ]:
df = pd.read_csv("data/concrete.csv")
print(df.shape)
df.head()

In [ ]:
# Confira os nomes reais das colunas e ajuste aqui se necessário.
# O dataset clássico (Yeh) tem: cement, slag, ash, water, superplasticizer,
# coarse_agg, fine_agg, age, e o alvo (resistência em MPa).
print(df.columns.tolist())
df.info()
df.describe()

In [ ]:
# Ajuste o nome da coluna-alvo conforme o que aparecer acima.
TARGET = "csMPa"  # <-- troque se o nome real for diferente

assert TARGET in df.columns, f"Coluna alvo '{TARGET}' não encontrada. Colunas disponíveis: {df.columns.tolist()}"
df.isna().sum()

## 2. EDA rápida

Só o suficiente para entender a distribuição do alvo e correlações entre features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], kde=True, ax=axes[0])
axes[0].set_title("Distribuição da resistência à compressão")

sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1])
axes[1].set_title("Correlação entre variáveis")
plt.tight_layout()
plt.show()

## 3. Split treino/teste + normalização

Separamos um conjunto de teste fixo, que só será usado no final (não entra no K-Fold).
A padronização (StandardScaler) é ajustada **apenas no treino**, para evitar vazamento de dados.

In [ ]:
X = df.drop(columns=[TARGET]).values.astype(np.float32)
y = df[TARGET].values.astype(np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)

scaler_X = StandardScaler().fit(X_train)
X_train_scaled = scaler_X.transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Também vale normalizar o alvo (ajuda a MLP a convergir); desnormalizamos na hora de avaliar.
scaler_y = StandardScaler().fit(y_train)
y_train_scaled = scaler_y.transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print("Treino:", X_train_scaled.shape, " Teste:", X_test_scaled.shape)

## 4. Dataset / Modelo MLP em PyTorch

In [ ]:
class ConcreteDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class MLP(nn.Module):
    def __init__(self, in_features, hidden=(64, 32), dropout=0.15):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 5. Treino com early stopping

Função reutilizável: treina até `max_epochs` ou até a validação parar de melhorar
por `patience` épocas seguidas (evita overfitting e economiza tempo).

In [ ]:
def train_model(model, train_loader, val_loader, max_epochs=300, patience=20, lr=1e-3, verbose=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(criterion(model(xb), yb).item())
        val_loss = np.mean(val_losses)

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and epoch % 20 == 0:
            print(f"Época {epoch:3d} | val_loss (MSE, escala normalizada): {val_loss:.4f}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping na época {epoch}")
            break

    model.load_state_dict(best_state)
    return model, best_val_loss

## 6. Validação cruzada (K-Fold) no conjunto de treino

Cada fold treina um modelo do zero e mede o RMSE na parte de validação (na escala original,
já desnormalizada). Isso dá uma estimativa mais robusta do desempenho do que um único split.

In [ ]:
def run_kfold(X, y, n_splits=5, hidden=(64, 32), dropout=0.15, batch_size=32):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rmses = []

    for fold, (tr_idx, val_idx) in enumerate(kfold.split(X)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        train_loader = DataLoader(ConcreteDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(ConcreteDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

        model = MLP(in_features=X.shape[1], hidden=hidden, dropout=dropout).to(device)
        model, _ = train_model(model, train_loader, val_loader)

        model.eval()
        with torch.no_grad():
            preds_scaled = model(torch.tensor(X_val, dtype=torch.float32).to(device)).cpu().numpy()

        # Desnormaliza para calcular RMSE em MPa (unidade real)
        preds = scaler_y.inverse_transform(preds_scaled)
        y_val_real = scaler_y.inverse_transform(y_val)

        rmse = np.sqrt(mean_squared_error(y_val_real, preds))
        fold_rmses.append(rmse)
        print(f"Fold {fold+1}/{n_splits} — RMSE: {rmse:.3f} MPa")

    print(f"\nRMSE médio (K-Fold): {np.mean(fold_rmses):.3f} ± {np.std(fold_rmses):.3f} MPa")
    return fold_rmses

fold_rmses = run_kfold(X_train_scaled, y_train_scaled, n_splits=5)

## 7. Modelo final e avaliação no conjunto de teste

Depois de validar a arquitetura via K-Fold, treinamos um modelo final usando todo o
treino (com uma fatia interna para early stopping) e avaliamos só uma vez no teste.

In [ ]:
# Fatia interna do treino só para o early stopping do modelo final
X_fit, X_val_final, y_fit, y_val_final = train_test_split(
    X_train_scaled, y_train_scaled, test_size=0.15, random_state=SEED
)

train_loader = DataLoader(ConcreteDataset(X_fit, y_fit), batch_size=32, shuffle=True)
val_loader = DataLoader(ConcreteDataset(X_val_final, y_val_final), batch_size=32, shuffle=False)

final_model = MLP(in_features=X_train_scaled.shape[1], hidden=(64, 32), dropout=0.15).to(device)
final_model, _ = train_model(final_model, train_loader, val_loader, verbose=True)

final_model.eval()
with torch.no_grad():
    test_preds_scaled = final_model(torch.tensor(X_test_scaled, dtype=torch.float32).to(device)).cpu().numpy()

test_preds = scaler_y.inverse_transform(test_preds_scaled)
y_test_real = scaler_y.inverse_transform(y_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test_real, test_preds))
mae = mean_absolute_error(y_test_real, test_preds)
r2 = r2_score(y_test_real, test_preds)

print(f"RMSE: {rmse:.3f} MPa")
print(f"MAE:  {mae:.3f} MPa")
print(f"R²:   {r2:.3f}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test_real, test_preds, alpha=0.6)
lims = [min(y_test_real.min(), test_preds.min()), max(y_test_real.max(), test_preds.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Resistência real (MPa)")
plt.ylabel("Resistência prevista (MPa)")
plt.title("Previsto vs. Real — conjunto de teste")
plt.tight_layout()
plt.show()

## Próximos passos

- Baseline com **XGBoost/LightGBM** para comparar com a MLP.
- Análise de importância de variáveis com **SHAP**.
- Se a MLP for competitiva: etapa de otimização (Grid Search / algoritmo genético) para sugerir formulações.